# EDA — Camada Bronze

Análise exploratória simples sobre o dado **bruto** gravado pelos extractors (`src/extractors/`) — sem nenhuma normalização ainda. Lê direto dos arquivos JSON particionados por `ano=/mes=/data_extracao=`, no backend configurado em `.env` (`BRONZE_STORAGE_BACKEND=local` ou `hdfs`).

Objetivo: olhar o dado como ele chega da fonte — schema real, nulos, duplicidade aparente, formatos de data — antes de qualquer regra de `src/transformers/rules.py` entrar em ação.


In [ ]:
import sys
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Notebook roda a partir de notebooks/ — a raiz do projeto é o diretório pai.
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

load_dotenv(dotenv_path=project_root / ".env")

pd.set_option("display.max_columns", 40)
print(f"Project root: {project_root}")


## 1. Carregar uma fonte da Bronze

Reaproveita `src/extractors/storage.py` (mesma camada que os extractors usam para gravar) para listar e ler os arquivos — funciona igual em `local` ou `hdfs`, sem duplicar lógica de leitura aqui.


In [ ]:
import os

from src.extractors import storage as bronze_storage


def load_bronze_source(source: str, limit_files: int | None = 8) -> pd.DataFrame:
    """Concatena arquivos JSON de uma fonte da Bronze em um único DataFrame.

    `limit_files` mantém a exploração leve (a Bronze real de `empenhos` tem
    centenas de chunks) — None lê o histórico inteiro.
    """
    base = f"{bronze_storage.BRONZE_BASE_PATH.rstrip('/')}/{source}"
    if bronze_storage.BRONZE_BACKEND == "hdfs":
        client = bronze_storage._get_hdfs_client()
        paths = [
            f"{dirpath}/{name}"
            for dirpath, _dirs, files in client.walk(base)
            for name in files
            if name.endswith(".json")
        ]
    else:
        paths = [
            os.path.join(dirpath, name).replace(os.sep, "/")
            for dirpath, _dirs, files in os.walk(base)
            for name in files
            if name.endswith(".json")
        ]
    paths = sorted(paths)
    if limit_files:
        paths = paths[:limit_files]

    prefix = bronze_storage.BRONZE_BASE_PATH.replace("\\", "/").rstrip("/") + "/"
    frames = []
    for path in paths:
        rel = path[len(prefix):] if path.startswith(prefix) else path
        frames.append(pd.DataFrame(bronze_storage.read_json_records(rel)))
    return pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()


print(f"Backend: {bronze_storage.BRONZE_BACKEND} · base: {bronze_storage.BRONZE_BASE_PATH}")


In [ ]:
df_contratos = load_bronze_source("contratos")
df_empenhos = load_bronze_source("empenhos")

print("contratos:", df_contratos.shape)
print("empenhos:", df_empenhos.shape)


## 2. Contratos (API) — schema, nulos e datas


In [ ]:
df_contratos.head()


In [ ]:
print(df_contratos.dtypes)
print()
print("% nulo por coluna (top 15):")
print((df_contratos.isna().mean() * 100).round(1).sort_values(ascending=False).head(15))


In [ ]:
# Formato real da API é DD/MM/YYYY, não ISO — confere aqui antes de qualquer normalização.
df_contratos["data_assinatura"].dropna().sample(min(5, df_contratos["data_assinatura"].notna().sum()), random_state=0)


In [ ]:
print("Modalidades mais frequentes:")
df_contratos["descricao_modalidade"].value_counts().head(10)


In [ ]:
# Possível duplicidade de `id` dentro da amostra carregada (esperado até o dedup da Silver).
dups = df_contratos["id"].duplicated().sum()
print(f"IDs duplicados na amostra: {dups} de {len(df_contratos)}")


## 3. Empenhos (PostgreSQL) — schema, valores e datas


In [ ]:
df_empenhos.head()


In [ ]:
print(df_empenhos.dtypes)
print()
print("% nulo por coluna (top 15):")
print((df_empenhos.isna().mean() * 100).round(1).sort_values(ascending=False).head(15))


In [ ]:
# Colunas de data chegam como TEXT ('YYYY-MM-DD HH:MM:SS.mmm'), não DATE/TIMESTAMP.
df_empenhos["dataemissao"].dropna().sample(min(5, df_empenhos["dataemissao"].notna().sum()), random_state=0)


In [ ]:
df_empenhos["valor"] = pd.to_numeric(df_empenhos["valor"], errors="coerce")
df_empenhos["valor"].describe()


In [ ]:
# empenhos não tem PRIMARY KEY na origem — (id, ano) é a chave lógica usada no dedup da Silver.
dups = df_empenhos.duplicated(subset=["id", "ano"]).sum()
print(f"Duplicatas em (id, ano) na amostra: {dups} de {len(df_empenhos)}")


## Achados rápidos

- Datas vêm em formatos diferentes por fonte (`DD/MM/YYYY` na API, texto ISO com hora no Postgres) — é isso que `src/transformers/rules.py` normaliza antes da Silver.
- Nenhuma tabela de origem tem `PRIMARY KEY` — `id` sozinho não garante unicidade (`empenhos` repete `id` entre anos diferentes).
- Próximo passo natural: comparar esta amostra bruta com a mesma fonte já na Silver (`eda_silver.ipynb`) para ver o efeito da normalização e do `MERGE INTO`.
